In [56]:
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table, Column, String, ForeignKey, DateTime, Integer, Double
from pathlib import Path

In [61]:
tables = [
    'customers',
    'products',
    'sellers',
    'orders',
    'order_items',
    'order_payments',
    'order_reviews'
]

In [19]:
for t in tables:
    print(t)
    df = pd.read_csv(f'data/{t}.csv')
    df.info()
    print('\n')

customers
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


products
<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty   

In [22]:
for t in tables:
    print(t)
    df = pd.read_csv(f'data/{t}.csv')
    print(df.describe())
    print('\n')

customers
       customer_zip_code_prefix
count              99441.000000
mean               35137.474583
std                29797.938996
min                 1003.000000
25%                11347.000000
50%                24416.000000
75%                58900.000000
max                99990.000000


products
       product_name_lenght  product_description_lenght  product_photos_qty  \
count         32341.000000                32341.000000        32341.000000   
mean             48.476949                  771.495285            2.188986   
std              10.245741                  635.115225            1.736766   
min               5.000000                    4.000000            1.000000   
25%              42.000000                  339.000000            1.000000   
50%              51.000000                  595.000000            1.000000   
75%              57.000000                  972.000000            3.000000   
max              76.000000                 3992.000000           20

In [38]:
for t in tables:
    print(t)
    df = pd.read_csv(f'data/{t}.csv')
    print(df.isnull().sum())
    print('\n')

customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


products
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


sellers
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


order_items
order_id              

In [57]:
#Remove data from products table that aren't required for current EDA
df = pd.read_csv('data/products.csv')
df = df.drop(columns=['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 
                      'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'])

In [58]:
#Fill null values in product_category name with 'Unknown'
df = df.fillna('Unknown')
df.isnull().sum()

product_id               0
product_category_name    0
dtype: int64

In [59]:
#Save new products table to a new csv file
raw_file = Path('data/products.csv')
raw_file.rename('data/raw_products.csv')
df.to_csv('data/products.csv', index=False)

In [73]:
#Modifying date format in orders table to remove time for simplicity.
df = pd.read_csv('data/orders.csv')

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col]).dt.date

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02,2017-10-02,2017-10-04,2017-10-10,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24,2018-07-26,2018-07-26,2018-08-07,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08,2018-08-08,2018-08-08,2018-08-17,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18,2017-11-18,2017-11-22,2017-12-02,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13,2018-02-13,2018-02-14,2018-02-16,2018-02-26


In [74]:
#Save new products table to a new csv file
raw_file = Path('data/orders.csv')
raw_file.rename('data/raw_orders.csv')
df.to_csv('data/orders.csv', index=False)

In [47]:
#The null values in orders table represent orders that have never been delivered either due to being canceled,
#stuck in processing or various other reasons. Therefore this is crucial data that doesn't need to be modified.
df = pd.read_csv('data/orders.csv')
df[df.isna().any(axis=1)]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaN,2018-06-28 00:00:00
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaN,NaN,2018-08-21 00:00:00
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaN,NaN,2017-10-03 00:00:00
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaN,2018-02-07 00:00:00
...,...,...,...,...,...,...,...,...
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00
99313,e9e64a17afa9653aacf2616d94c005b8,b4cd0522e632e481f8eaf766a2646e86,processing,2018-01-05 23:07:24,2018-01-09 07:18:05,NaN,NaN,2018-02-06 00:00:00
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaN,NaN,NaN,2018-09-27 00:00:00
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaN,NaN,2017-09-15 00:00:00


In [77]:
#Connet to postgresql
username = 'postgres'
password = 'ResiNotEvil1997-1'
host = 'localhost'
port = '5432'
database = 'OrderDelivery'

engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}')

In [78]:
#Create empty tables, defining primary keys and foreign keys
metadata = MetaData()

customers = Table(
    "customers",
    metadata,
    Column('customer_id', String(50), primary_key=True),
    Column('customer_unique_id', String(50)),
    Column('customer_zip_code_prefix', Integer),
    Column('customer_city', String(50)),
    Column('customer_state', String(50))
)
order_items = Table(
    "order_items",
    metadata,
    Column('order_id', String(50), ForeignKey('orders.order_id')),
    Column('order_item_id', Integer),
    Column('product_id', String(50), ForeignKey('products.product_id')),
    Column('seller_id', String(50), ForeignKey('sellers.seller_id')),
    Column('shipping_limit_date', DateTime),
    Column('price', Double),
    Column('freight_value', Double)
)
order_payments = Table(
    "order_payments",
    metadata,
    Column('order_id', String(50), ForeignKey('orders.order_id')),
    Column('payment_sequential', Integer),
    Column('payment_type', String(50)),
    Column('payment_installments', Integer),
    Column('payment_value', Double)
)
order_reviews = Table(
    "order_reviews",
    metadata,
    Column('review_id', String(50)),
    Column('order_id', String(50), ForeignKey('orders.order_id')),
    Column('review_score', Integer),
    Column('review_creation_date', DateTime),
    Column('review_answer_timestamp', DateTime)
)
orders = Table(
    "orders",
    metadata,
    Column('order_id', String(50), primary_key=True),
    Column('customer_id', String(50), ForeignKey('customers.customer_id')),
    Column('order_status', String(50)),
    Column('order_purchase_timestamp', DateTime),
    Column('order_approved_at', DateTime),
    Column('order_delivered_carrier_date', DateTime),
    Column('order_delivered_customer_date', DateTime),
    Column('order_estimated_delivery_date', DateTime)
)
products = Table(
    "products",
    metadata,
    Column('product_id', String(50), primary_key=True),
    Column('product_category_name', String(50)),
    Column('product_name_lenght', Integer),
    Column('product_description_lenght', Integer),
    Column('product_photos_qty', Integer),
    Column('product_weight_g', Integer),
    Column('product_length_cm', Integer),
    Column('product_height_cm', Integer),
    Column('product_width_cm', Integer)
)
sellers = Table(
    "sellers",
    metadata,
    Column('seller_id', String(50), primary_key=True),
    Column('seller_zip_code_prefix', Integer),
    Column('seller_city', String(50)),
    Column('seller_state', String(50))
)
metadata.create_all(engine)


In [79]:
#Fill table with data
for t in tables:
    df = pd.read_csv(f'data/{t}.csv')
    df.to_sql(t, engine, if_exists='append', index=False)
